# Assetwise Data

In [1]:
import json
import requests
from pathlib import Path
from requests.auth import HTTPBasicAuth

In [2]:
from civilpy.state.ohio.DOT.AssetWise import get_assetwise_secrets

username, password = get_assetwise_secrets()

# Get a List of Bridges to Query

In [3]:
from src.civilpy.state.ohio.DOT.TIMS import get_bridge_sfns_by_district

In [5]:
d2_bridges = get_bridge_sfns_by_district(2)

Querying API for District 02...
URL: https://tims.dot.state.oh.us/ags/rest/services/Assets/Bridge_Inventory/MapServer/0/query
Filter: DISTRICT = '02'

Success! Found 3740 features.


In [6]:
d2_bridges

['2600137',
 '2600250',
 '2600463',
 '2601540',
 '2601710',
 '2626454',
 '2629070',
 '2629380',
 '2629402',
 '2629666',
 '2630052',
 '2630141',
 '2630168',
 '2600145',
 '2600196',
 '2600277',
 '2600439',
 '2600544',
 '2600722',
 '2601141',
 '2601184',
 '2601591',
 '2601621',
 '2630176',
 '2630303',
 '2630613',
 '2631076',
 '2631334',
 '2629097',
 '2629348',
 '2629569',
 '2629623',
 '2629984',
 '2631512',
 '2631555',
 '2631644',
 '2631903',
 '2632268',
 '2630001',
 '2630044',
 '2630338',
 '2630532',
 '2630621',
 '2630818',
 '2631067',
 '2631083',
 '2631326',
 '2631520',
 '2632578',
 '2632683',
 '2632810',
 '2632926',
 '2632985',
 '2631598',
 '2631857',
 '2631946',
 '2632373',
 '2632756',
 '2633086',
 '2633477',
 '2633523',
 '2633752',
 '2633841',
 '2633221',
 '2633302',
 '2633337',
 '2633515',
 '2633647',
 '2634163',
 '2634309',
 '2634333',
 '2634619',
 '2634630',
 '2633736',
 '2633744',
 '2633957',
 '2634287',
 '2634376',
 '2634902',
 '2635232',
 '2635283',
 '2635739',
 '2660016',
 '26

In [3]:
# from civilpy.state.ohio.DOT.SNBI import get_all_bridges_paged

def get_assets_paged(
    starting_row: int = 0,
    items_per_page: int = 100,
    api_type: str = "api",
    include_coordinates: bool = False,
    include_parent: bool = False
):
    """
    Retrieves a paged list of assets from the AssetWise API using a POST request.

    Args:
        starting_row (int): The starting row for the page (0-indexed). Defaults to 0.
        items_per_page (int): The number of items to return per page. Defaults to 100.
        api_type (str): The API type, either "api" or "mobile". Defaults to "api".
        include_coordinates (bool): Whether to include asset coordinates in the response. Defaults to False.
        include_parent (bool): Whether to include the parent as_id in the response. Defaults to False.

    Returns:
        dict: A dictionary containing the API response data for assets, including pagination info.
    
    Raises:
        requests.exceptions.HTTPError: If the API request returns a non-200 status.
    """
    username, password = get_assetwise_secrets()
    
    api_url = f"https://ohiodot-it-api.bentley.com/{api_type}/Asset/GetAssets"

    # Query parameters (IncludeCoordinates and IncludeParent are query params for both GET and POST)
    query_params = {
        "IncludeCoordinates": include_coordinates,
        "IncludeParent": include_parent
    }

    headers = {
        "Accept": "application/json",
        "Content-Type": "application/json" # Essential for sending JSON in the request body
    }
    
    # Request body for pagination, matching the RequestPaging schema
    request_body = {
        "starting": starting_row,
        "count": items_per_page
    }

    print(f"Requesting URL: {api_url} with query params: {query_params}, body: {request_body}")
    
    response = requests.post(
        api_url,
        params=query_params,
        headers=headers,
        auth=HTTPBasicAuth(username, password),
        json=request_body # Send pagination data as JSON in the body
    )
    
    response.raise_for_status() # Raise an exception for HTTP errors
    
    print("Successfully retrieved paged assets.")
    return response.json()

In [4]:
all_bridges = []
current_starting_row = 0
page_size = 2000  # 4000 + records at a time seem to break the data limits
total_assets = None 

while True:
    print(f"Fetching assets starting from row {current_starting_row} with page size {page_size}...")
    assets_data = get_assets_paged(
        starting_row=current_starting_row,
        items_per_page=page_size,
        include_coordinates=True,
        include_parent=True
    )

    if total_assets is None:
        # Get total_assets from the first successful response
        total_assets = assets_data.get('totalCount')
        print(f"Total assets available (approx): {total_assets}")

    if assets_data and assets_data.get('data'):
        current_page_count = len(assets_data['data'])
        for asset in assets_data['data']:
            all_bridges.append(asset)
        print(f"Retrieved {current_page_count} assets from this page. Current total collected: {len(all_bridges)}")

        # Check if there are more pages to fetch
        if len(all_bridges) >= total_assets: # or if current_page_count < page_size
            print("All assets collected.")
            break
        
        # Move to the next page
        current_starting_row += page_size
    else:
        print("No more data received or an empty page was returned.")
        break

print(f"\nFinal count of all bridges collected: {len(all_bridges)}")

Fetching assets starting from row 0 with page size 2000...
Requesting URL: https://ohiodot-it-api.bentley.com/api/Asset/GetAssets with query params: {'IncludeCoordinates': True, 'IncludeParent': True}, body: {'starting': 0, 'count': 2000}
Successfully retrieved paged assets.
Total assets available (approx): 46324
Retrieved 2000 assets from this page. Current total collected: 2000
Fetching assets starting from row 2000 with page size 2000...
Requesting URL: https://ohiodot-it-api.bentley.com/api/Asset/GetAssets with query params: {'IncludeCoordinates': True, 'IncludeParent': True}, body: {'starting': 2000, 'count': 2000}
Successfully retrieved paged assets.
Retrieved 2000 assets from this page. Current total collected: 4000
Fetching assets starting from row 4000 with page size 2000...
Requesting URL: https://ohiodot-it-api.bentley.com/api/Asset/GetAssets with query params: {'IncludeCoordinates': True, 'IncludeParent': True}, body: {'starting': 4000, 'count': 2000}
Successfully retrieved

In [ ]:
import pandas as pd

print("Converting collected bridge data to a DataFrame...")
bridges_df = pd.DataFrame(all_bridges)

# Handle nested 'coordinates' dictionary
# First, ensure 'coordinates' column exists and contains dictionaries
if 'coordinates' in bridges_df.columns:
    # Create new columns for each coordinate field by applying a lambda function
    # The .get() method is used for safe access, returning None if a key doesn't exist.
    # The isinstance(x, dict) check handles cases where 'coordinates' might not be a dictionary (e.g., None).
    bridges_df['latitude'] = bridges_df['coordinates'].apply(lambda x: x.get('latitude') if isinstance(x, dict) else None)
    bridges_df['longitude'] = bridges_df['coordinates'].apply(lambda x: x.get('longitude') if isinstance(x, dict) else None)
    bridges_df['latitude_Degree'] = bridges_df['coordinates'].apply(lambda x: x.get('latitude_Degree') if isinstance(x, dict) else None)
    bridges_df['latitude_Minute'] = bridges_df['coordinates'].apply(lambda x: x.get('latitude_Minute') if isinstance(x, dict) else None)
    bridges_df['latitude_Seconds'] = bridges_df['coordinates'].apply(lambda x: x.get('latitude_Seconds') if isinstance(x, dict) else None)
    bridges_df['longitude_Degree'] = bridges_df['coordinates'].apply(lambda x: x.get('longitude_Degree') if isinstance(x, dict) else None)
    bridges_df['longitude_Minute'] = bridges_df['coordinates'].apply(lambda x: x.get('longitude_Minute') if isinstance(x, dict) else None)
    bridges_df['longitude_Second'] = bridges_df['coordinates'].apply(lambda x: x.get('longitude_Second') if isinstance(x, dict) else None)
    bridges_df['usesDegMinSec'] = bridges_df['coordinates'].apply(lambda x: x.get('usesDegMinSec') if isinstance(x, dict) else None)
    
    # Drop the original 'coordinates' column as its contents have been flattened
    bridges_df = bridges_df.drop(columns=['coordinates'])

bridges_df

# Get All Bridges for a Specific District

In [34]:
district = 6

In [6]:
single_bridge = bridges_df[bridges_df['as_code'] == '0702870']

In [7]:
single_bridge

,as_id,as_name,as_code,as_guid,rt_id,as_parent_id,as_parent,as_asset_def,as_root,as_deleted,...,assetType,latitude,longitude,latitude_Degree,latitude_Minute,latitude_Seconds,longitude_Degree,longitude_Minute,longitude_Second,usesDegMinSec
3195,69795,BEL-00070-2134 _(0702870),0702870,6eedcf92-f0aa-49ec-b664-8ed2714ce95e,1,20636,None,0,False,False,...,None,40.068261,-80.842869,40.0,4.0,5.0,-80.0,-50.0,-34.0,False


For a single value you can also more efficiently directly query the API,

In [8]:
def get_asset_by_code(
    asset_code: str,
    include_coordinates: bool = False,
    include_parent: bool = False
) -> dict:
    """
    Retrieves a specific asset by its asset code using the AssetWise API.

    Args:
        asset_code (str): The unique code of the asset to retrieve.
                          (e.g., '0702870' for a bridge) [1].
        api_type (str): The API type, typically 'api' or 'mobile'.
                        Defaults to 'api' [1].
        include_coordinates (bool): Whether to include asset coordinates
                                    in the response. Defaults to False [1].
        include_parent (bool): Whether to include the parent as_id in the
                               response. Defaults to False [1].

    Returns:
        dict: A dictionary representing the API response, which
              should conform to the ApiAssetSuccessReturn schema upon
              success [2].

    Raises:
        requests.exceptions.HTTPError: If the API request
                                       encounters an HTTP error (e.g.,
                                       400 Bad Request, 401 Unauthorized,
                                       404 Not Found) [2-4].
    """
    username, password = get_assetwise_secrets()

    # The base URL for the API server [5]
    base_url = "https://ohiodot-it-api.bentley.com"

    # Construct the API URL for GetAssetByAsCode,
    # embedding apiType and as_code as path parameters [1]
    api_url = (
        f"{base_url}/api/Asset/GetAssetByAsCode/{asset_code}"
    )

    # Query parameters (IncludeCoordinates and IncludeParent are
    # query parameters for this endpoint) [1]
    query_params = {
        "IncludeCoordinates": include_coordinates,
        "IncludeParent": include_parent
    }

    headers = {
        "Accept": "application/json"
    }

    print(f"Requesting URL: {api_url} with query params: {query_params}")

    # Make the GET request
    response = requests.get(
        api_url,
        params=query_params,
        headers=headers,
        auth=HTTPBasicAuth(username, password)
    )

    # Raise an exception for HTTP errors (4xx or 5xx)
    response.raise_for_status()

    print("Successfully retrieved asset by code.")
    return response.json()['data']

In [9]:
bridge = get_asset_by_code('0702870', include_coordinates=True, include_parent=True)

Requesting URL: https://ohiodot-it-api.bentley.com/api/Asset/GetAssetByAsCode/0702870 with query params: {'IncludeCoordinates': True, 'IncludeParent': True}
Successfully retrieved asset by code.


In [10]:
bridge

{'as_id': 69795,
 'as_name': 'BEL-00070-2134 _(0702870)',
 'as_code': '0702870',
 'as_guid': '6eedcf92-f0aa-49ec-b664-8ed2714ce95e',
 'rt_id': 1,
 'as_parent_id': 20636,
 'as_parent': None,
 'as_asset_def': 0,
 'coordinates': {'usesDegMinSec': False,
  'latitude': '40.068261',
  'longitude': '-80.842869',
  'latitude_Degree': 40.0,
  'latitude_Minute': 4.0,
  'latitude_Seconds': 5.0,
  'longitude_Degree': -80.0,
  'longitude_Minute': -50.0,
  'longitude_Second': -34.0},
 'as_root': False,
 'as_deleted': False,
 'at_id': 1,
 'as_child_rt_id': 1,
 'as_child_at_id': None,
 'as_summary': False,
 'as_is_asset_view': False,
 'asset_status_id': 4,
 'as_corridor': False,
 'as_last_snapshot_date': None,
 'as_is_internal': False,
 'as_sub_asset_parent': True,
 'as_last_update_date': '2025-11-05T06:28:14.01',
 'as_default_file_set_date': '2024-05-30T14:20:29.233',
 'as_stage_pontis': False,
 'as_federal_submission_type': 1,
 'asset_segment_map': None,
 'corridor': None,
 'as_created_date': '2020-

## API Parameters

Various additional values need to be determined before being utilized in  
the api,

*   **`apiType`** [1, 6, 8, etc.]: This variable is nearly ubiquitous across the API paths. It is described as "All requests should be api" and is **required**. Its schema indicates it must be a `string` matching the pattern `^api$|^mobile$`, with a `default` value of "api". This means you'd typically use either `/api/` or `/mobile/` in your URLs.
*   **`act_guid`**: Represents the **ActivityType GUID**. It is a **required** path parameter for getting an ActivityType by its GUID. Its schema defines it as a `string` with a `uuid` format.
*   **`act_id`**: Represents the **ActivityType ID**. It is a **required** path parameter used to get an ActivityType by its ID, its name, or in various `AssetActivityTypeSchedule` and `InspectionReportInspTypeMap` operations. Its schema defines it as an `integer` with `int32` format.
*   **`td_id`**: Represents the **TaskDefinition ID**. It is a **required** path parameter used to retrieve ActivityTypes, TaskDefinitions, or WorkFlows by TaskDefinition ID. Its schema defines it as an `integer` with `int32` format.
*   **`as_id`**: Represents the **Asset ID**. This is a very common **required** parameter for fetching specific assets, reports, schedules, files, current values, and performing various asset-related updates or deletions. Its schema defines it as an `integer` with `int32` format.
*   **`as_guid`**: Represents the **Asset GUID**. It is a **required** path parameter for retrieving specific assets or asset files by their globally unique identifier. Its schema defines it as a `string` with `uuid` format.
*   **`as_code`**: Represents the **Asset Code**. It is a **required** path parameter to get an asset by its code or access level by asset code. Its schema defines it as a `string`.
*   **`sch_id`**: Represents the **Schedule ID**. This is a **required** path parameter for retrieving or manipulating asset activity type schedules. Its schema defines it as an `integer` with `int32` format.
*   **`as_sch_freq_length`**: Represents the **Schedule Frequency Length**. It is a **required** path parameter for updating the frequency unit length of open activities. Its schema defines it as an `integer` with `int32` format.
*   **`as_sch_freq_unit`**: Represents the **Schedule Frequency Unit**. It is a **required** path parameter indicating units like Hours, Days, Weeks, Months, or Years, used in schedule updates. Its schema is a reference to `#/components/schemas/ScheduleFrequencyUnit` and lists `x-enumNames` for clarity.
*   **`previous_as_sch_freq_unit`**: Represents the **Previous Schedule Frequency Unit**. It is a **required** path parameter used when updating overridden frequency units. Its schema is the same as `as_sch_freq_unit`.
*   **`af_id`**: Represents the **AssetFile ID**. It is a **required** path parameter for operations related to asset files such as getting, deleting, or mapping them. Its schema defines it as an `integer` with `int32` format.
*   **`include_binary`**: A **required** boolean flag in some `AssetFile` paths to indicate whether to include the file binary in the response. It defaults to `false`.
*   **`include_file_type`**: A **required** boolean flag in some `AssetFile` paths to indicate whether to include the file type in the response. It defaults to `false`.
*   **`include_asset_file_categories`**: A **required** boolean flag in some `AssetFile` paths. Its default value is `false`.
*   **`object_id`**: Represents the **Object ID**. This is a **required** parameter for various calls that interact with different types of objects (Asset, Report, Segment, FormGroup, Form, Field). Its schema defines it as an `integer` with `int32` format.
*   **`object_type`**: Represents the **Object Type**. This is a **required** parameter that specifies the type of object being referenced. It uses a `ObjectType` schema which is an enum with values like `0 = Asset`, `1 = Report`, `4 = Segment`, `5 = FormGroup`, `6 = Form`, `7 = Field`.
*   **`file_name`**: Represents the **File Name**. It is a **required** path parameter for retrieving an asset's map. Its schema defines it as a `string`.
*   **`affm_guid`**: Represents the **AssetFilesFieldMap GUID**. It is a **required** path parameter to get a map by its GUID. Its schema defines it as a `string` with `uuid` format.
*   **`instance`**: Represents an **Instance number**. It is a **required** path parameter for deleting an `AssetFilesFieldMap`. Its schema defines it as an `integer` with `int32` format.
*   **`sub_as_id`**: Represents the **Sub-Asset ID**. It is a **required** path parameter in paths like `GetMapsForReportForSubAsset`. Its schema defines it as an `integer` with `int32` format.
*   **`child_ast_id`**: Represents the **Child Asset Task ID**. It is a **required** path parameter in paths like `GetMapsForChildReportForSummaryReport`. Its schema defines it as an `integer` with `int32` format.
*   **`summary_ast_guid`**: Represents the **Summary Asset Task GUID**. It is a **required** path parameter in paths like `GetMapsForChildReportForSummaryReport` or `UnlinkChildReportFromSummaryReport`. Its schema defines it as a `string` with `uuid` format.
*   **`fe_id`**: Represents the **Field ID**. This is a very common **required** parameter used across many entities like AssetFilesFieldMap, CurrentValue, FieldChoice, FormElement, RoleFieldSecurity, and Value. Its schema defines it as an `integer` with `int32` format.
*   **`old_ast_id`**: Represents the **Old Asset Task ID**. It is a **required** path parameter for copying asset files. Its schema defines it as an `integer` with `int32` format.
*   **`new_ast_id`**: Represents the **New Asset Task ID**. It is a **required** path parameter for copying asset files. Its schema defines it as an `integer` with `int32` format.
*   **`wfs_id`**: Represents the **Workflow Stage ID**. This is a **required** parameter for various `AssetFilesWorkflowMap`, `AssetTask`, and `WorkflowStage` operations. Its schema defines it as an `integer` with `int32` format.
*   **`old_wfs_id`**: Represents the **Old Workflow Stage ID**. It is a **required** path parameter for updating an `AssetFilesWorkflowMap`. Its schema defines it as an `integer` with `int32` format.
*   **`new_wfs_id`**: Represents the **New Workflow Stage ID**. It is a **required** path parameter for updating an `AssetFilesWorkflowMap`. Its schema defines it as an `integer` with `int32` format.
*   **`description_fe_id`**: Represents the **Description Field ID**. It is a **required** path parameter for getting asset segments with fields. Its schema defines it as an `integer` with `int32` format.
*   **`includeCompleted`**: A **required** boolean flag for `AssetSegment/GetAssetSegmentsWithFields`.
*   **`geometryAsGeoJSON`**: A **required** boolean flag for `AssetSegment/GetAssetSegmentsWithFields`.
*   **`segmentOrder`**: Represents the **Segment Order**. It is a **required** path parameter for getting a specific asset segment. Its schema defines it as an `integer` with `int32` format.
*   **`asset_status_id`**: Represents the **Asset Status ID**. It is a **required** path parameter to get an AssetStatus by its ID. Its schema defines it as an `integer` with `int32` format.
*   **`AssetStatusGuid`**: Represents the **Asset Status GUID**. It is a **required** path parameter to get an AssetStatus by its GUID. Its schema defines it as a `string` with `uuid` format.
*   **`ast_guid`**: Represents the **Asset Task GUID**. This is a common **required** parameter for various report, asset task, and project work specification operations. Its schema defines it as a `string` with `uuid` format.
*   **`at_guid`**: Represents the **AssetType GUID**. It is a **required** path parameter to get an AssetType by its GUID. Its schema defines it as a `string` with `uuid` format.
*   **`at_name`**: Represents the **AssetType Name**. It is a **required** path parameter to get an AssetType by name or in combination with `dt_id` for DataTypeMembers. Its schema defines it as a `string`.
*   **`at_id`**: Represents the **AssetType ID**. It is a **required** path parameter to get an AssetType by its ID, manage role mappings, or retrieve element definitions and report types associated with it. Its schema defines it as an `integer` with `int32` format.
*   **`rl_id`**: Represents the **Role ID**. It is a **required** path parameter used for managing asset type role maps, inspection type role maps, and retrieving user roles. Its schema defines it as an `integer` with `int32` format.
*   **`currentValueGuid`**: Represents the **Current Value GUID**. It is a **required** path parameter to get a current value by its GUID. Its schema defines it as a `string` with `uuid` format.
*   **`instanceCount`**: Represents the **Instance Count**. It is a **required** path parameter for retrieving DataType values up to a specific instance count. Its schema defines it as an `integer` with `int32` format.
*   **`dt_key`**: Represents the **DataType Key**. It is a **required** path parameter for DataTypeInstance operations like getting, swapping, or clearing values. Its schema defines it as a `string`.
*   **`dt_id`**: Represents the **DataType ID**. It is a **required** path parameter for various DataType and DataTypeInstance operations. Its schema defines it as an `integer` with `int32` format.
*   **`tm_id`**: Represents the **Type Member ID**. It is a **required** path parameter used for DataTypeInstanceMember and DataTypeMember operations. Its schema defines it as an `integer` with `int32` format.
*   **`ti_id`**: Represents the **Type Instance ID**. It is a **required** path parameter used for DataTypeInstanceMember operations. Its schema defines it as an `integer` with `int32` format.
*   **`eg_id`**: Represents the **Element Group ID**. It is a **required** path parameter for retrieving element definitions by element group and asset type. Its schema defines it as an `integer` with `int32` format.
*   **`parent_sub_as_id`**: Represents the **Parent Sub Asset ID**. It is a **required** path parameter for getting elements for a parent. Its schema defines it as an `integer` with `int32` format.
*   **`ed_id`**: Represents the **Element Definition ID**. It is a **required** path parameter for getting an element definition ID. Its schema defines it as an `integer` with `int32` format.
*   **`reportable_as_id`**: Represents the **Reportable Asset ID**. It is a **required** path parameter for getting an element definition ID. Its schema defines it as an `integer` with `int32` format.
*   **`ede_list_id`**: Represents the **Element List Definition ID**. It is a **required** path parameter for getting the count of elements in a list. Its schema defines it as an `integer` with `int32` format.
*   **`ede_id`**: Represents the **Element Definition Element ID**. It is a **required** path parameter for getting the count of elements in a list. Its schema defines it as an `integer` with `int32` format.
*   **`enumName`**: Represents the **Enum Name**. It is a **required** path parameter to get an enum by its name. Its schema defines it as a `string`.
*   **`fe_guid`**: Represents the **Field GUID**. It is a **required** path parameter to get a Field by its GUID. Its schema defines it as a `string` with `uuid` format.
*   **`fe_name`**: Represents the **Field Name**. It is a **required** path parameter to get a Field by its name. Its schema defines it as a `string`.
*   **`fm_id`**: Represents the **Form ID**. This is a common **required** parameter for retrieving form elements, mobile forms, or managing form security. Its schema defines it as an `integer` with `int32` format.
*   **`va_value`**: Represents a **Value**. It is a **required** path parameter for getting a FieldChoice by ID and value. Its schema defines it as a `string`.
*   **`parent_fc_id`**: Represents the **Parent Field Choice ID**. It is a **required** path parameter for getting FieldChoices based on a parent. Its schema defines it as an `integer` with `int32` format.
*   **`fc_id`**: Represents the **Field Choice ID**. It is a **required** path parameter for getting, updating, or deleting a FieldChoice by its ID. Its schema defines it as an `integer` with `int32` format.
*   **`feId`**: Represents the **Field ID** (alias for fe_id). It is a **required** path parameter for returning certified bridge inspector field choices. Its schema defines it as an `integer` with `int32` format.
*   **`tp_id`**: Represents the **Template ID**. It is a **required** path parameter for retrieving elements by template or data type template display settings. Its schema defines it as an `integer` with `int32` format.
*   **`it_id`**: Represents the **Inspection Type ID**. This is a **required** parameter for operations involving inspection type schedules, report inspection type maps, and workflows. Its schema defines it as an `integer` with `int32` format.
*   **`lockRpt`**: Represents a boolean flag for **locking or unlocking a report**. It is a **required** path parameter. Its schema defines it as a `boolean`.
*   **`complete_date`**: Represents the **Completion Date**. It is a **required** path parameter used to mark a report inspection type map as complete. Its schema defines it as a `string` with `date-time` format.
*   **`maint_ast_id`**: Represents the **Maintenance Item Asset Task ID**. It is a **required** path parameter for checking if a maintenance item instance is linked with an open inspection report. Its schema defines it as an `integer` with `int32` format.
*   **`it_guid`**: Represents the **InspectionType GUID**. It is a **required** path parameter to get an InspectionType by its GUID. Its schema defines it as a `string` with `uuid` format.
*   **`platform`**: Represents the **Platform Type**. It is a **required** path parameter for getting inspection types with their given platform template. It uses a `PlatformType` schema, which is an enum (`0 = Web`, `1 = Mobile`, `2 = Output`).
*   **`in_id`**: Represents the **Inspector ID** or **User ID**. This is a **required** parameter for retrieving inspection types for a user, roles mapped to a user, or user security details. Its schema defines it as an `integer` with `int32` format.
*   **`ws_id`**: Represents the **Working Set ID**. It is a path parameter to remove a working set by ID. Its schema defines it as an `integer` with `int32` format.
*   **`form_display_order`**: Represents the **Form Display Order**. It is a **required** path parameter for updating the display order of a form. Its schema defines it as an `integer` with `int32` format.
*   **`fg_id`**: Represents the **Form Group ID**. This is a common **required** path parameter for managing mobile report type form maps and report type form group maps. Its schema defines it as an `integer` with `int32` format.
*   **`fg_order`**: Represents the **Form Group Order**. It is a **required** path parameter for updating the display order of a form group. Its schema defines it as an `integer` with `int32` format.
*   **`rt_id`**: Represents the **Report Type ID**. This is a very common **required** parameter for operations related to report types, including getting specific types, form maps, inspection type maps, and roles. Its schema defines it as an `integer` with `int32` format.
*   **`et_id`**: Represents the **Element Type ID**. It is a **required** path parameter for getting asset metadata. The documentation doesn't specify its type directly within the path, but by context of other IDs it is likely an integer.
*   **`el_col_binding`**: Represents the **Element Column Binding**. It is a **required** path parameter for retrieving object metadata. Its schema defines it as an `integer` with `int32` format.
*   **`ObjectName`**: Represents the **Object Name**. It is a **required** path parameter for getting properties of an object. Its schema defines it as a `string`.
*   **`filter`**: A path parameter for filtering properties of an object. It's marked as `allowEmptyValue: true`.
*   **`parent_task_ast_guid`**: Represents the **Parent/Project Task Asset GUID**. It is a **required** path parameter for retrieving project work specification details for a user. Its schema defines it as a `string` with `uuid` format.
*   **`getAll`**: A **required** boolean path parameter for `ReportType/GetAll`, indicating whether to get all report types or only those for the current user. It defaults to `false`.
*   **`id_type`**: Represents the **ID Type**. It is a **required** path parameter for `ReportTypeAssetTypeMap` operations. It uses an `IdType` schema, which is an enum (`0 = AssetType`, `1 = ReportType`).
*   **`id`**: Represents an **ID**. It is a **required** path parameter for `ReportTypeAssetTypeMap` operations. Its schema defines it as an `integer` with `int32` format.
*   **`AsDefault`**: A **required** boolean path parameter for `ReportTypeInspectionTypeMap/Delete`, defaulting to `false`.
*   **`sch_name`**: Represents the **Schedule Task Definition Name**. It is a **required** path parameter to get a schedule task definition by name. Its schema defines it as a `string`.
*   **`exact`**: A **required** boolean path parameter for `ScheduleTaskDefinitions/GetScheduleTaskDefinitionByName`, defaulting to `true`.
*   **`sch_guid`**: Represents the **Schedule Task Definition GUID**. It is a **required** path parameter to get a schedule task definition by its GUID. Its schema defines it as a `string` with `uuid` format.
*   **`ade_id`**: Represents the **Agency Developed Element ID**. It is a **required** path parameter for getting ADE mappings. Its schema defines it as an `integer` with `int32` format.
*   **`se_id`**: Represents the **Structure Element ID**. This is a **required** parameter for various `StructElementADEMap`, `StructureElementAssetTypeMap`, and `StructureElementProtectiveSystemMap` operations. Its schema defines it as an `integer` with `int32` format.
*   **`StructElementId`**: Represents the **Structure Element ID**. It is a **required** path parameter for getting protective systems for a structure element. Its schema defines it as an `integer` with `int32` format.
*   **`seg_as_id`**: Represents the **Segment Asset ID**. It is a **required** path parameter for getting segments of an object. Its schema defines it as an `integer` with `int32` format.
*   **`asId`**: Represents the **Asset ID** within `StructureElement` paths. It is a **required** path parameter. Its schema defines it as an `integer` with `int32` format.
*   **`segmentId`**: Represents the **Segment ID**. It is a path parameter for `StructureElement/GetElements`, marked as `allowEmptyValue: true` and defaults to `0`. Its schema defines it as an `integer` with `int32` format.
*   **`elementId`**: Represents the **Element ID**. It is a path parameter for `StructureElement/GetElements`, marked as `allowEmptyValue: true` and defaults to `0`. Its schema defines it as an `integer` with `int32` format.
*   **`environmentId`**: Represents the **Environment ID**. It is a path parameter for `StructureElement/GetElements`, marked as `allowEmptyValue: true` and defaults to `0`. Its schema defines it as an `integer` with `int32` format.
*   **`defect_se_id`**: Represents the **Defect Structure Element ID**. It is a **required** path parameter for `StructureElementDefectMap` operations. Its schema defines it as an `integer` with `int32` format.
*   **`ps_se_id`**: Represents the **Protective System Structure Element ID**. It is a **required** path parameter for `StructureElementProtectiveSystemMap` operations. Its schema defines it as an `integer` with `int32` format.
*   **`sm_id`**: Represents the **Structure Material ID**. It is a **required** path parameter to get a specific material by ID. Its schema defines it as an `integer` with `int32` format.
*   **`ast_type`**: Represents the **Asset Task Type**. It is a **required** path parameter for getting TaskDefinitions. It uses an `AssetTaskType` schema, which is an enum including types like `WorkOrder`, `FeasibleAction`, `ProjectTask`, `ProjectWorkSpec`, and `SuppWorkSpec`.
*   **`ug_id`**: Represents the **User Group ID**. It is a **required** path parameter for getting users for a user group. Its schema defines it as an `integer` with `int32` format.
*   **`sub_asset_id`**: Represents the **Sub-Asset ID**. It is a **required** path parameter for getting values by sub-asset ID. Its schema defines it as an `integer` with `int32` format.
*   **`wf_id`**: Represents the **WorkFlow ID**. This is a **required** parameter for retrieving or manipulating WorkFlows and WorkflowStages. Its schema defines it as an `integer` with `int32` format.
*   **`wf_type`**: Represents the **Workflow Type**. It is a **required** path parameter for getting WorkFlows by type. It uses a `WorkflowType` schema, which is an enum (`0 = InspectionReports`, `1 = Maintenance`).
*   **`wfa_id`**: Represents the **Workflow Action ID**. It is a **required** path parameter for updating workflow actions or getting possible workflow stages. Its schema defines it as an `integer` with `int32` format.

Methods for getting the most important of these values is demonstrated  
below.

Let's start with getting all inspection images/files for a structure,

## Images/Files

In [11]:
import requests
import pandas as pd
from requests.auth import HTTPBasicAuth
from typing import List, Dict, Union

def get_asset_report_files_metadata(
    asset_id: int,
    api_type: str = "api",
    include_asset_file: bool = True
) -> List[Dict[str, Union[int, str, bool, Dict]]]:
    """
    Retrieves metadata for files associated with a specific asset that are
    also mapped to reports.

    Args:
        asset_id (int): The unique ID of the asset (e.g., a bridge).
        api_type (str): The API type, typically 'api'. Defaults to 'api' [5].
        include_asset_file (bool): Whether to include the Asset File
                                    information (including thumbnail) within
                                    the response. Defaults to True.

    Returns:
        List[Dict]: A list of dictionaries, where each dictionary represents
                    an AssetFilesReportMap object, containing file metadata
                    like af_id, af_filename, and a nested 'assetFile' object
                    [1, 6].

    Raises:
        requests.exceptions.HTTPError: If the API request encounters an HTTP error.
    """
    username, password = get_assetwise_secrets()
    base_url = "https://ohiodot-it-api.bentley.com"
    api_url = (
        f"{base_url}/{api_type}/AssetFilesReportMap/GetAssetFilesForAsset/{asset_id}"
    )

    query_params = {
        "IncludeAssetFile": include_asset_file
    }

    headers = {
        "Accept": "application/json"
    }

    print(
        f"Requesting URL: {api_url} with query params: {query_params}"
    )

    response = requests.get(
        api_url,
        params=query_params,
        headers=headers,
        auth=HTTPBasicAuth(username, password)
    )
    response.raise_for_status() # Raise an exception for HTTP errors [7-19]

    print(f"Successfully retrieved file metadata for asset {asset_id}.")
    return response.json().get('data', []) # The 'data' field contains the list [20]